# LSTM avanzado con PyTorch Lightning y TensorBoard

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion1/4-lstm-avanzado-con-pytorch-lightning.ipynb)

Este notebook conserva una versión más completa del trabajo con LSTM para clasificación de noticias en español. A diferencia del notebook mínimo de esta misma sesión, aquí mantenemos utilidades de experimentación como PyTorch Lightning, TensorBoard, early stopping y una organización del entrenamiento más cercana a un flujo de trabajo real.

**Sugerencia pedagógica:** úsalo como material de profundización después de recorrer `3-embeddings-y-lstm-minimo.ipynb`.

#### Referencias
- Dataset: https://huggingface.co/datasets/mteb/spanish_news
- [Long Short-Term Memory](https://www.researchgate.net/publication/13853244_Long_Short-Term_Memory#fullTextFileContent)
- [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)


In [1]:
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets

### Cargando el dataset
Este es un dataset pequeño de articulos de noticias en idioma español con sus respectivas categorías. El dataset está disponible en el HuggingFace Hub y puede ser fácilmente descargado con la librería.

In [3]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('mteb/spanish_news', split='train')
dataset

Dataset({
    features: ['language', 'label', 'newspaper', 'hash', 'text'],
    num_rows: 8099
})

Observemos uno de sus registros

In [4]:
dataset[1]

{'language': 'es',
 'label': 9,
 'newspaper': 'bbc_news',
 'hash': '48c2c042c74264da3c623bad78b5e9ca75531207',
 'text': 'Fuente de la imagen, AFPUn boceto de la cápsula espacial CST-100 de Boeing.La agencia espacial estadounidense NASA anunció este martes que firmó un contrato con las empresas privadas Boeing y SpaceX (por montos de US$4.200 millones y US$2.600 millones respectivamente) para la construcción de nuevas naves espaciales que transporten astronautas a la Estación Espacial Internacional.La noticia fue comunicada por el administrador de la NASA, Charles Bolden, quien explicó que se han impuesto el ambicioso objetivo de 2017 para llevar a cabo el primer lanzamiento desde territorio estadounidense desde que culminó el programa de transbordadores en 2011.Durante los últimos tres años, los astronautas estadounidenses han debido depender de cohetes rusos Soyuz, a un costo de más de US$70 millones de dólares por asiento.Los lanzamientos, según el anuncio, se harán de nuevo desde el

Para los efectos de esta tarea, nos servirán el texto y la categoría naturalmente.

A manera general, observemos que tan largos o cortos tienden a ser los textos.

In [5]:
text_lengths = [len(row['text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

Texto más corto: 501
Texto más largo: 204324
Longitud promedio: 4244.560192616373


Estos valores son la cantidad de *caractéres* que tiene las secuencias. Una decisión ingenua pero útil en este momento podría ser ajustar la longitud de las secuencias que vamos a usar para el entrenamiento a unos 2000 tokens. Esto podría ser suficiente para capturar una porción significativa de los textos.

## Definiendo el Tokenizer

Ahora, vamos a definir el tokenizer para nuestra tarea. Para mantener las cosas simples, vamos a mantener un conteo de palabras y vamos a hacer un corte hasta los primeros 50mil tokens.

In [6]:
import re
from collections import Counter

def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-záéíóúüñ]+", " ", text)
    return text.strip().split()

# Construimos el vocabulario a partir de conjunto de datos.
token_counts = Counter()
for text in dataset["text"]:
    token_counts.update(simple_tokenizer(text))

# 50k-2 porque necesitamos reservar espacio para los dos tokens especiales
top_n_tokens = list(token_counts.keys())[:50000-2]
vocab = {"[PAD]": 0, "[UNK]": 1}
for token in top_n_tokens:
    vocab[token] = len(vocab)

def tokenize_text(text, max_length=50):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(tok, vocab["[UNK]"]) for tok in tokens[:max_length]]
    ids += [vocab["[PAD]"]] * (max_length - len(ids))
    return ids

Exploremos ahora el tokenizador obtenido.

In [7]:
print(f"Vocabulario: {len(vocab)} tokens")
print("Primeros 15 tokens:")
print(f"{top_n_tokens[:15]}")
print("15 tokens de en medio:")
print(f"{top_n_tokens[1000:1015]}")
print("Últimos 15 tokens:")
print(f"{top_n_tokens[-15:]}")

Vocabulario: 50000 tokens
Primeros 15 tokens:
['una', 'de', 'las', 'novedades', 'que', 'google', 'introdujo', 'en', 'sus', 'nuevos', 'pixel', 'y', 'luego', 'ha', 'llevado']
15 tokens de en medio:
['wisconsin', 'traspasado', 'pacers', 'cortado', 'franquicia', 'indiana', 'opciones', 'válidas', 'agente', 'libre', 'bayern', 'múnich', 'pablo', 'laso', 'reencuentro']
Últimos 15 tokens:
['mavi', 'icrea', 'liset', 'prida', 'proselitismo', 'corporativos', 'convencerles', 'filmarla', 'legisle', 'igualará', 'ingenua', 'jugase', 'aprendiera', 'superarnos', 'resultarle']


Esta forma de exploración es para darnos una idea de las palabras más utilizadas en el corpus y nos dará un indicio de si la tokenización es adecuada o no. Vemos que tenemos algunos stop words, como artículos (el, la) y conectores (del, que). Para una tarea de clasificación de texto podríamos prescindir de estos pero para facilitar las cosas y ya que los demás tokens lucen bien, podemos preservarlos.

Ahora veamos como convierte el tokenizador una oración muy sencilla:

In [8]:
tokenized = tokenize_text("hola mundo", max_length=8)
tokenized

[4698, 517, 0, 0, 0, 0, 0, 0]

Lo que obtenemos de vuelta son los ids de cada token según el vocabulario. Ahora algo importante que notamos aquí es el *padding*, durante el entrenamiento, queremos que las secuencias sean de tamaño fijo, para asi operar comodamente con matrices. Pero ya vimos que no todos los textos tienen la misma longitud. Entonces que hacer? para los que son más largos que una longitud dada simplemente cortamos, pero para los que son más cortos, debemos *rellenar* lo faltante con un *token especial de relleno o padding*. Y es justo lo que definimos allí, cuando la cadena es inferior a 8 **tokens**, entonces debemos hacer padding hasta que se cumplan los 8.

Si queremos saber a que token exactamente hacen referencia estos ids, simplemente revisamos el vocabulario que hemos construido:

In [9]:
id_2_token = {v: k for k, v in vocab.items()}
[id_2_token[token] for token in tokenized]

['hola', 'mundo', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

Claramente vemos los 3 tokens como cadenas independientes (el padding se considera un token independiente).

### Definiendo el dataset de pytorch
Ahora podemos proceder a definir el dataset. Esto debería ser muy sencillo dado que nuestro dataset es pequeño y ya tenemos el tokenizador listo.

In [10]:
import torch
import numpy as np
from typing import Tuple, Dict
from torch.utils.data import Dataset

class SpanishNewsDataset(Dataset):

    def __init__(self, tokenizer, dataset, seq_length: int = 512):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.seq_length = seq_length
        # Definimos estos dos mapas para facilitarnos la tarea
        # de traducir de nombres de categoría a ids de categoría.
        self.id_2_class_map = dict(enumerate(np.unique(dataset[:]['label'])))
        self.class_2_id_map = {v: k for k, v in self.id_2_class_map.items()}
        self.num_classes = len(self.id_2_class_map)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        text, y = self.dataset[index]['text'], self.dataset[index]['label']
        y = self.class_2_id_map[y]
        data = {'input_ids': torch.tensor(self.tokenizer(text, max_length=self.seq_length))}
        data['y'] = torch.tensor(y)
        return data


    def __len__(self):
        return len(self.dataset)

Ahora instanciaremos el dataset entero. Para este experimento, definiremos un tamaño máximo de secuencia de 2048 **tokens**. Que según nuestra intuición arriba, debería ser suficiente para la tarea.

In [11]:
max_len = 512 
spanish_news_dataset = SpanishNewsDataset(tokenize_text, dataset, seq_length=max_len)
assert len(spanish_news_dataset) == len(dataset)

Y luego, procedemos a hacer el train-val-test split y crear los dataloaders.

In [12]:
from torch.utils.data import random_split
from torch.utils.data import DataLoader

batch_size = 4 if not IN_COLAB else 16
train_dataset, val_dataset, test_dataset = random_split(spanish_news_dataset, lengths=[0.8, 0.1, 0.1])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

## Definición del modelo LSTM

Ahora vamos a configurar un módulo pytorch simple para este problema. Vamos ha utilizar los embeddings, que vendrían siendo los vectores de palabra. Pytorch nos ofrece una capa con la que directamente podemos entrenarlos a partir de los token ids obtenidos. El resto consistirá en invocar una capa LSTM seguida de una capa densa para la clasificación.

Recordemos que las redes recurrentes como las LSTM por diseño enlazan todas las dimensiones del vector de entrada, formando así la secuencia, la estructura natural que necesitamos representar.

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMBlock(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, _) = self.lstm(embedded)
        return hidden[-1]


### Definición del clasificador

Finalmente, definimos el modelo en si. Este modelo constará de 3 capas:

- La tokenización, tal como la definimos anteriormente.
- El bloque LSTM, que acabamos de decinir.
- Una capa densa adicional que servirá como clasificador de aquello que nos entregue la capa del transformer.

Como este es un LightningModule, aquí definiremos el resto de funciones utilitarias para el entrenamiento de la tarea.

In [14]:
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy

class SpanishNewsClassifierWithLSTM(LightningModule):

    def __init__(self, vocab_size: int, num_classes: int, emb_dim: int, hidden_dim: int = 128):
        super(SpanishNewsClassifierWithLSTM, self).__init__()
        self.num_classes = num_classes
        self.lstm = LSTMBlock(vocab_size, emb_dim, hidden_dim, num_classes)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
            nn.LogSoftmax(dim=1)
        )

        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc = Accuracy(task='multiclass', num_classes=num_classes)

    def forward(self, x):
        embeddings = self.lstm(x)
        return self.classifier(embeddings)

    
    def training_step(self, batch, batch_idx):
        x, y = batch['input_ids'], batch['y']
        # print(f"\nbatch-idx: {batch_idx}")
        # print(f"shape of x: {x.shape}")
        # print(torch.max(x, dim=0))
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.train_acc(y_hat, y)
        self.log('train-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train-acc', self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def validation_step(self, batch):
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.val_acc(y_hat, y)
        self.log('val-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val-acc', self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def test_step(self, batch):
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        self.test_acc(y_hat, y)
        self.log('test-acc', self.test_acc, prog_bar=True, on_step=False, on_epoch=True)


    def predict_step(self, batch):
        x = batch['input_ids']
        return self(x)


    def configure_optimizers(self):
        optimizer =  torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-5)
        return optimizer

    
model = SpanishNewsClassifierWithLSTM(vocab_size=len(vocab) + 1, num_classes=spanish_news_dataset.num_classes, emb_dim=256)

tb_logger = TensorBoardLogger('tb_logs', name='LSTMClassifier')
callbacks=[EarlyStopping(monitor='train-loss', patience=3, mode='min')]
trainer = Trainer(max_epochs=10, devices=1, logger=tb_logger, callbacks=callbacks, precision="16-mixed")

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm       │ LSTMBlock          │ 13.1 M │ train │     0 │
│ 1 │ classifier │ Sequential         │  200 K │ train │     0 │
│ 2 │ train_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_acc    │ MulticlassAccuracy │      0 │ train │     0 │
│ 4 │ test_acc   │ MulticlassAccuracy │      0 │ train │     0 │
└───┴────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 13.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 13.3 M                                                                                               
Total estimated model params size (MB): 53.322                                                                     
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=10` reached.


Observemos el proceso de entrenamiento

In [15]:
%load_ext tensorboard

In [16]:
%tensorboard --logdir tb_logs/

Y como es de esperarse, realizaremos la validación contra el conjunto de prueba.

In [17]:
model.eval()
trainer.test(model, test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test-acc          │    0.8529047966003418     │
└───────────────────────────┴───────────────────────────┘

[{'test-acc': 0.8529047966003418}]

### Haciendo predicciones

Finalmente, vamos a hacer uso del modelo y ver que tan bueno es para la clasificación de noticias.

In [18]:
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
predictions = torch.argmax(predictions, dim=-1)
predictions = [spanish_news_dataset.id_2_class_map[pred] for pred in predictions.numpy()]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


In [19]:
import pandas as pd

test_indices = test_dataset.indices
df = pd.DataFrame(data={
    "texto": dataset[test_indices]['text'],
    "tokens": [tokenize_text(v) for v in dataset[test_indices]['text']],
    "categoría": dataset[test_indices]['label'],
    'predicción': predictions
}, index=test_indices)

df['tokens_string'] = df.tokens.apply(lambda t: ' '.join([id_2_token[i] for i in t]))
df = df[["texto", "tokens", "tokens_string", "categoría", "predicción"]]
df.style.set_table_styles(
    [
        {'selector': 'td', 'props': [('word-wrap', 'break-word')]}
    ]
)
df.head(15)

,texto,tokens,tokens_string,categoría,predicción
6545,"En Dune, el recurso natural más valioso de la ...","[9, 44131, 41, 1346, 4979, 197, 15273, 3, 22, ...",en dune el recurso natural más valioso de la g...,4,4
4798,El obispo Franz-Peter Tebartz-van Elst ha sido...,"[41, 4128, 1, 14894, 1, 926, 1, 15, 261, 28175...",el obispo [UNK] peter [UNK] van [UNK] ha sido ...,10,10
5409,La sopa de pescado es uno de los grandes clási...,"[22, 10866, 3, 3469, 21, 1031, 3, 18, 687, 799...",la sopa de pescado es uno de los grandes clási...,6,6
7442,Seguramente recuerdes esa mítica secuencia de...,"[3408, 1, 266, 7819, 17511, 3, 1, 1, 2009, 17,...",seguramente [UNK] esa mítica secuencia de [UNK...,8,8
3936,"El técnico del Athletic, Ernesto Valverde, señ...","[41, 4512, 146, 24302, 17318, 4601, 4489, 377,...",el técnico del athletic ernesto valverde señal...,2,2
269,Por Julieta Villar 16 de febrero de 2024 / 12:...,"[55, 11149, 9972, 3, 893, 3, 23386, 4246, 3, 6...",por julieta villar de febrero de pmel arzobisp...,10,10
5280,Una buena tortilla de patatas es apetecible en...,"[2, 2295, 15596, 3, 15417, 21, 2505, 9, 658, 1...",una buena tortilla de patatas es apetecible en...,6,6
1932,Se trata de los ‘radares STOP’ de los que ya o...,"[39, 492, 3, 18, 6213, 19834, 3, 18, 6, 63, 19...",se trata de los radares stop de los que ya os ...,5,5
6638,La novena jornada de protestas de los agricult...,"[22, 3786, 2149, 3, 2056, 3, 18, 10416, 104, 4...",la novena jornada de protestas de los agricult...,4,4
2082,"No tiene nada que ver con 'Priscilla', pero lo...","[59, 171, 229, 6, 596, 34, 1, 30, 31, 197, 872...",no tiene nada que ver con [UNK] pero lo más cu...,1,1


In [20]:
errors = df[df['categoría'] != df['predicción']]
errors.head(15)

,texto,tokens,tokens_string,categoría,predicción
2085,Las principales materias primas de la cerveza ...,"[4, 281, 25448, 25449, 3, 22, 15959, 424, 22, ...",las principales materias primas de la cerveza ...,6,10
2235,Enlace copiadoEl Tribunal Supremo ha anulado e...,"[6102, 21370, 1348, 6031, 15, 4501, 41, 15018,...",enlace copiadoel tribunal supremo ha anulado e...,5,4
4529,Esta semana se ha presentado en la Casa del Hu...,"[419, 1950, 39, 15, 7433, 9, 22, 1114, 146, 20...",esta semana se ha presentado en la casa del hu...,10,4
3575,Cualquiera que haya viajado alguna vez a la ca...,"[136, 6, 1408, 33470, 2521, 606, 17, 22, 2057,...",cualquiera que haya viajado alguna vez a la ca...,6,10
2604,Por Jonah McKeown 15 de febrero de 2024 / 04:4...,"[55, 1, 1, 3, 893, 3, 1, 35, 20365, 444, 4718,...",por [UNK] [UNK] de febrero de [UNK] un tiroteo...,10,6
7260,"Las rebajas de verano están aquí. Así es, la e...","[4, 2762, 3, 715, 211, 616, 237, 21, 22, 16198...",las rebajas de verano están aquí así es la esp...,8,0
4524,La modelo Zoe Sozo Bethel ganó el año pasado M...,"[22, 36, 48686, 1, 1, 847, 41, 102, 711, 27415...",la modelo zoe [UNK] [UNK] ganó el año pasado m...,8,0
6891,"Forbes, marca referente en negocios y 'lifesty...","[38369, 3098, 3664, 9, 15024, 13, 36033, 9, 41...",forbes marca referente en negocios y lifestyle...,4,7
6043,Las fascias del brazo y del nivel del antebraz...,"[4, 1, 146, 10913, 13, 146, 120, 146, 37341, 2...",las [UNK] del brazo y del nivel del antebrazo ...,11,7
822,Se presentó en Alemania a finales de octubre y...,"[39, 7994, 9, 3396, 17, 3398, 3, 901, 13, 63, ...",se presentó en alemania a finales de octubre y...,5,0


## Conclusiones

- En este caso tenemos una tarea de clasificación de texto de múltiples clases.
- Estamos usando un bloque LSTM como featurizer, es decir lo usamos para extraer features de las secuencias de entrada con las cuales harémos predicciones luego.
- Nótese que de las capas LSTM, solo nos interesa la última, ya que esta recupera todas las operaciones enalazadas anteriores.
- Observamos que el modelo toma su tiempo en entrenar, esto es natural debido al diseño de las LSTM, donde por cada paso de tiempo se debe computar un gradiente, por lo que el computo es mucho mayor.